# VitiGuard MD — фильтр «лист / не лист» на полевых снимках (ml/fine-01)

Прежний фильтр отсеивал снимки куста с несколькими листьями как «не виноград». Блокнот заново обучает только фильтр (модель болезней не меняется):

- скачивает открытые наборы под CC BY 4.0: PlantVillage, NGLD, HERMOS, Tardif et al. (Flavescence dorée и эска в поле), Imagenette;
- берёт ряды INESC из `Мой диск/vitiguard/vine/vine_dataset.zip`;
- сравнивает новый фильтр со старым на отложенных снимках и пишет результат в `Мой диск/vitiguard/gate`.

**Перед запуском:** положите на Google Диск три файла модели с сервера: `diagnosis.onnx`, `diagnosis.json`, `diagnosis.gate.json` (лучше в `Мой диск/vitiguard/models`, но блокнот найдёт их в любой папке). Если их нет на Диске, блокнот попросит выбрать их с компьютера. GPU не нужен. «Среда выполнения» → «Выполнить все». Около 40 минут, большая часть — скачивание (около 7 ГБ).

In [ ]:
import glob
import os
import shutil

from google.colab import drive, files

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/vitiguard"
OUT = f"{DRIVE}/gate"
MODEL = "/content/model"
BRANCH = "main"
NEED = ["diagnosis.onnx", "diagnosis.json", "diagnosis.gate.json"]
os.makedirs(MODEL, exist_ok=True)

# Модель ищется по всему Диску: папку models могли положить не в vitiguard
found = sorted(glob.glob("/content/drive/MyDrive/**/diagnosis.onnx", recursive=True))
folder = os.path.dirname(found[0]) if found else None
if all(os.path.exists(f"{MODEL}/{name}") for name in NEED):
    print("Модель уже загружена в этой сессии")
elif folder and all(os.path.exists(f"{folder}/{name}") for name in NEED):
    print("Модель найдена на Диске:", folder)
    for name in NEED:
        shutil.copy(f"{folder}/{name}", f"{MODEL}/{name}")
else:
    print("На Диске модели нет. Выберите три файла с компьютера:", ", ".join(NEED))
    for name, data in files.upload().items():
        with open(f"{MODEL}/{os.path.basename(name)}", "wb") as file:
            file.write(data)

missing = [name for name in NEED if not os.path.exists(f"{MODEL}/{name}")]
assert not missing, f"Не хватает файлов модели: {missing}"
assert os.path.exists(f"{DRIVE}/vine/vine_dataset.zip"), "Нет Мой диск/vitiguard/vine/vine_dataset.zip"
print("Файлы модели на месте")


In [ ]:
# Повторный запуск: нельзя удалять папку, находясь в ней
%cd /content
!rm -rf /content/VitiGuard-MD_back
!git clone -q --depth 1 --branch {BRANCH} https://github.com/androtschii/VitiGuard-MD_back /content/VitiGuard-MD_back
%cd /content/VitiGuard-MD_back/ml
# Без зависимостей: numpy, Pillow, pydantic и PyYAML в Colab уже есть
!pip install -q --no-deps -e .
!pip install -q onnx onnxruntime

import sys

# Пакет, поставленный в уже запущенную сессию, она сама не видит
sys.path.insert(0, "/content/VitiGuard-MD_back/ml/src")


## Данные

Архивы распаковываются на диск Colab. Из больших полевых снимков сразу делаются копии до 1600 точек по длинной стороне: так хватает места и быстрее считать.

In [ ]:
import io
import random
import subprocess
import tarfile
import zipfile
from pathlib import Path

from PIL import Image

DATA = Path("/content/data")
DATA.mkdir(exist_ok=True)
SUFFIXES = (".jpg", ".jpeg", ".png", ".webp")

def fetch(url, target):
    """Скачивание через curl: Mendeley Data отвечает 403 клиенту Python без
    имени, а curl представляется сам собой."""
    target = Path(target)
    if not target.exists():
        print("скачивание", target.name, flush=True)
        target.parent.mkdir(parents=True, exist_ok=True)
        part = target.with_suffix(target.suffix + ".part")
        subprocess.run(
            ["curl", "-fL", "--retry", "3", "-sS", "-o", str(part), url], check=True
        )
        part.rename(target)
    return target


def frames_from_zip(source, out, limit=700, max_side=1600):
    """Случайные снимки из архива (и из вложенных архивов), уменьшенные копии."""
    out = Path(out)
    if out.exists() and any(out.iterdir()):
        print(out.name, "уже готово")
        return out
    out.mkdir(parents=True, exist_ok=True)
    count = 0

    def walk(zip_file):
        nonlocal count
        names = [n for n in zip_file.namelist() if "__macosx" not in n.lower()]
        photos = [n for n in names if n.lower().endswith(SUFFIXES)]
        random.Random(0).shuffle(photos)
        for name in photos:
            if count >= limit:
                return
            try:
                image = Image.open(io.BytesIO(zip_file.read(name))).convert("RGB")
            except Exception:
                continue
            image.thumbnail((max_side, max_side))
            image.save(out / f"{count:05d}.jpg", quality=85)
            count += 1
        for name in names:
            if count < limit and name.lower().endswith(".zip"):
                with zipfile.ZipFile(io.BytesIO(zip_file.read(name))) as inner:
                    walk(inner)

    with zipfile.ZipFile(source) as zip_file:
        walk(zip_file)
    print(out.name, count, "снимков")
    return out


MENDELEY = "https://data.mendeley.com/public-api/zip/{}/download/{}"

In [ ]:
# Листья по одному и листья других культур — PlantVillage, только нужные папки
GRAPE = ["Grape___Black_rot", "Grape___Esca_(Black_Measles)",
         "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)", "Grape___healthy"]
OTHER = ["Apple___healthy", "Apple___Apple_scab", "Tomato___healthy",
         "Tomato___Early_blight", "Corn_(maize)___healthy", "Potato___healthy",
         "Soybean___healthy", "Strawberry___healthy", "Cherry_(including_sour)___healthy"]
PV = DATA / "plantvillage"
if not PV.exists():
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset.git {PV}
folders = " ".join(f'"raw/color/{name}"' for name in GRAPE + OTHER)
!git -C {PV} sparse-checkout set {folders}

In [ ]:
from vitiguard_ml.data import hermos, ngld

# Архивы кладутся туда, где их ждёт код набора: повторно он их не качает
fetch(ngld.DOWNLOAD_URL, DATA / "ngld" / f"ngld-{ngld.VERSION}.zip")
fetch(hermos.DOWNLOAD_URL, DATA / "hermos" / f"hermos-{hermos.VERSION}.zip")
ngld_root = ngld.download(DATA / "ngld")
hermos_zip = hermos.download(DATA / "hermos")
rows, skipped = hermos.prepare(hermos_zip, DATA / "hermos_crops")
print("вырезки HERMOS:", len(rows))
frames_from_zip(hermos_zip, DATA / "canopy_hermos")
tardif_zip = fetch(MENDELEY.format("3dr9r3w3jn", 2), DATA / "tardif.zip")
frames_from_zip(tardif_zip, DATA / "canopy_tardif")
frames_from_zip(f"{DRIVE}/vine/vine_dataset.zip", DATA / "canopy_inesc")
imagenette = fetch("https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz", DATA / "imagenette.tgz")
if not (DATA / "imagenette2-160").exists():
    with tarfile.open(imagenette) as archive:
        archive.extractall(DATA)

## Обучение фильтра и сравнение со старым

In [ ]:
lab = " ".join(f'"{PV}/raw/color/{name}"' for name in GRAPE) + f' "{ngld_root}"'
other = " ".join(f'"crops={PV}/raw/color/{name}"' for name in OTHER)
!python -m vitiguard_ml.gate_field --model /content/model/diagnosis.onnx     --lab {lab} --crops {DATA}/hermos_crops     --canopy hermos={DATA}/canopy_hermos tardif={DATA}/canopy_tardif inesc={DATA}/canopy_inesc     --not-grape imagenette={DATA}/imagenette2-160 {other}     --out /content/gate

In [ ]:
import json
import shutil

os.makedirs(OUT, exist_ok=True)
for name in ["diagnosis.gate.json", "gate_report.json"]:
    shutil.copy(f"/content/gate/{name}", f"{OUT}/{name}")
report = json.loads(Path("/content/gate/gate_report.json").read_text())
keys = sorted({key for row in report.values() for key in row})
print(f"{'':<22}" + "".join(f"{name:>8}" for name in report))
for key in keys:
    print(f"{key:<22}" + "".join(f"{report[name].get(key, float('nan')):>8.3f}" for name in report))
print("Готово: скачайте папку Мой диск/vitiguard/gate")